In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-7I-_iCgmJM-p


In [4]:
# import v1.0
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# LLM 모델
# FewShotPromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate

## FewShotPromptTemplate 설계

In [10]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 파리
            언어 : 프랑스어
            음식 : 치즈와 와인
            통화 : 유료
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 도쿄
            언어 : 일본어
            음식 : 초밥과 라멘
            통화 : 엔
        """
    },
    {
        "country": "브라질에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 브라질리아
            언어 : 포르투갈어
            음식 : 페이조아다와 슈하스코
            통화 : 헤알
        """
    }
]

## 2단계: 예시용 프롬프트 템플릿 정의

In [11]:
example_template = """
    Human: {country}
    AI: {answer}
"""

example_prompt = PromptTemplate.from_template(example_template)

example_prompt

PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='\n    Human: {country}\n    AI: {answer}\n')

## 3단계 : FewShotPromptTemplate 생성 후 결합 (예시를 전달하는 템플릿)

In [14]:
prompt = FewShotPromptTemplate(
    # 질문
    example_prompt=example_prompt,
    # 예시
    examples=examples,
    # 사용자의 질문
    suffix="Human : {country}에 대해서 어떻게 알고 있어?",
    input_variables=["country"]
)

In [15]:
prompt

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'country': '프랑스에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도 : 파리\n            언어 : 프랑스어\n            음식 : 치즈와 와인\n            통화 : 유료\n        '}, {'country': '일본에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도 : 도쿄\n            언어 : 일본어\n            음식 : 초밥과 라멘\n            통화 : 엔\n        '}, {'country': '브라질에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도 : 브라질리아\n            언어 : 포르투갈어\n            음식 : 페이조아다와 슈하스코\n            통화 : 헤알\n        '}], example_prompt=PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='\n    Human: {country}\n    AI: {answer}\n'), suffix='Human : {country}에 대해서 어떻게 알고 있어?')

In [17]:
chat = ChatOpenAI(temperature=0)

In [18]:
chain = prompt | chat

In [20]:
result = chain.invoke({
    "country" : "한국"
})

In [22]:
print(result.content)

AI:
        저는 이렇게 알고 있어요.
        수도 : 서울
        언어 : 한국어
        음식 : 김치와 불고기
        통화 : 대한민국 원


In [23]:
result = chain.invoke({
    "country" : "아프가니스탄"
})

In [24]:
print(result.content)

AI: 
            저는 이렇게 알고 있어요.
            수도 : 카불
            언어 : 파슈토어, 다리어
            음식 : 케밥, 볶음밥
            통화 : 아프가니스탄 애프간ي


## FewShotChatMessagePromptTemplate

In [25]:
# v1.0
# ChatModel
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

In [27]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 파리
            언어 : 프랑스어
            음식 : 치즈와 와인
            통화 : 유료
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 도쿄
            언어 : 일본어
            음식 : 초밥과 라멘
            통화 : 엔
        """
    },
    {
        "country": "브라질에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 브라질리아
            언어 : 포르투갈어
            음식 : 페이조아다와 슈하스코
            통화 : 헤알
        """
    }
]

## 2단계 예시용 프롬프트 템플릿 정의

In [31]:
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{country}에 대해서 어떻게 알고 있어?"),
    ("ai", "{answer}")
    ])

In [32]:
prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

In [34]:
# final = prompt | chain

In [41]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 박사입니다. 짧은 답변을 제공하세요."),
    example_prompt,
    ("human", "{country}에 대해 얼마나 알고 있어?")
])

In [45]:
final_chain = prompt | chain

In [48]:
print(final_prompt)

input_variables=['answer', 'country'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 지리학 박사입니다. 짧은 답변을 제공하세요.'), additional_kwargs={}), ChatPromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}에 대해서 어떻게 알고 있어?'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['answer'], input_types={}, partial_variables={}, template='{answer}'), additional_kwargs={})]), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}에 대해 얼마나 알고 있어?'), additional_kwargs={})]


In [52]:
result = final_chain.invoke([
    "country":"일본"
])

SyntaxError: invalid syntax (1289663656.py, line 2)

In [53]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 파리
            언어 : 프랑스어
            음식 : 치즈와 와인
            통화 : 유료
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 도쿄
            언어 : 일본어
            음식 : 초밥과 라멘
            통화 : 엔
        """
    },
    {
        "country": "브라질에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 브라질리아
            언어 : 포르투갈어
            음식 : 페이조아다와 슈하스코
            통화 : 헤알
        """
    }
]

In [54]:
example_prompt = PromptTemplate.from_template("Human: {country}\n AI: {answer}")

## Selector 연결

In [63]:
# 최신 버전에서는 아래 경로를 주로 사용합니다
from langchain_core.example_selectors import LengthBasedExampleSelector

# 그 후 기존 코드를 실행하세요
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=10
)

In [69]:
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    # 아래와 같이 input_variables를 명시해야 합니다.
    input_variables=["country"], 
    suffix="Human: {country}에 대해 어떻게 알고 있어?",
    # 만약 prefix도 사용 중이라면 함께 확인해 보세요.
    prefix="다양한 나라에 대해 설명하는 AI입니다."
)

In [74]:
print(prompt.format(**{
    "country": "브라질"
    }))

다양한 나라에 대해 설명하는 AI입니다.

Human: 브라질에 대해 어떻게 알고 있어?


In [75]:
final_chain = prompt | chat

In [76]:
final_chain = prompt | chat
result = final_chain.invoke({
    "country":"독일"
})

In [77]:
print(result.content)

AI: 독일은 중앙 유럽에 위치한 국가로, 유럽 연합의 주요 회원국 중 하나입니다. 독일은 세계적으로 경제적으로 강력한 국가로서 자동차 산업, 기계 공학, 화학 산업 등 다양한 산업 분야에서 선두를 달리고 있습니다. 또한 독일은 문화적으로도 다양한 면모를 가지고 있으며, 음악, 문학, 예술 등 다양한 분야에서 세계적으로 유명한 인물들을 배출해 왔습니다. 독일은 역사적으로도 중요한 역할을 해 왔으며, 두 차례의 세계 대전을 겪은 국가로서 평화와 화해를 추구하는 모습을 보여주고 있습니다.


## Chat 모델(LengthBasedExampleSelector)

In [79]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 파리
            언어 : 프랑스어
            음식 : 치즈와 와인
            통화 : 유료
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 도쿄
            언어 : 일본어
            음식 : 초밥과 라멘
            통화 : 엔
        """
    },
    {
        "country": "브라질에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 브라질리아
            언어 : 포르투갈어
            음식 : 페이조아다와 슈하스코
            통화 : 헤알
        """
    }
]

In [85]:
length_prompt = PromptTemplate(
    input_variavles=["country", "answer"],
    template="""
        Human: {country}에 대해서 어떻게 알고 있어요? \n
        AI: {answer}
    """
)

example_prompt = ChatPromptTemplate.from_messages([
    ("Human: {country}에 대해 어떻게 알고 있어?"),
    ("ai", "{answer}")
])

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=length_prompt,
    max_length=200
)

fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt
)

In [86]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 전문가입니다."),
    fewshot_prompt,
    ("human", "{country}에 대해 어떻게 알고 있나요?")
])

In [87]:
final_chain = prompt | chat
result = final_chain.invoke({
    "country":"독일"
})

In [88]:
print(result.content)

AI: 독일은 중앙 유럽에 위치한 국가로, 유럽 연합의 주요 회원국 중 하나입니다. 독일은 세계적으로 경제적으로 강력한 국가로서 자동차 산업, 기계 공학, 화학 산업 등 다양한 산업 분야에서 선두를 달리고 있습니다. 또한 독일은 문화적으로도 다양한 유산을 보유하고 있으며, 베를린을 비롯한 다양한 도시들은 역사적인 유적과 현대적인 문화 시설을 함께 갖추고 있습니다. 독일은 또한 유럽 내에서 중요한 정치적 역할을 하고 있으며, 유럽 연합의 중심 국가 중 하나로서 다양한 국제 기구와 협력하고 있습니다.
